In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install -q transformers accelerate peft datasets
!pip install -q -U torchao

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)

print("Base model loaded:", MODEL_NAME)
print("Params:", round(sum(p.numel() for p in base_model.parameters())/1e6, 1), "M")

In [ ]:
from peft import LoraConfig, get_peft_model
import copy
import random

LORA_RANK = 8
LORA_ALPHA = 16

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

toy_tasks = {
    "grammar_en": [
        ("Fix the grammar: She go to school every day.", "She goes to school every day."),
        ("Fix the grammar: He don't like apples.", "He doesn't like apples."),
        ("Fix the grammar: They was late.", "They were late."),
    ],
    "grammar_de": [
        ("Fix the grammar: Er gehen zur Schule.", "Er geht zur Schule."),
        ("Fix the grammar: Wir hat keine Zeit.", "Wir haben keine Zeit."),
        ("Fix the grammar: Sie ist mein Schwester.", "Sie ist meine Schwester."),
    ],
    "grammar_fr": [
        ("Fix the grammar: Il ne sais pas.", "Il ne sait pas."),
        ("Fix the grammar: Nous avons manger.", "Nous avons mangé."),
        ("Fix the grammar: Elle est aller au marché.", "Elle est allée au marché."),
    ],
}
print("Toy tasks defined:", list(toy_tasks.keys()))

In [ ]:
from torch.utils.data import Dataset

class ToyDataset(Dataset):
    def __init__(self, pairs, tokenizer, max_len=64):
        self.pairs = pairs
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        prompt, target = self.pairs[idx]
        text = f"{prompt}\nAnswer: {target}{self.tokenizer.eos_token}"
        enc = self.tokenizer(
            text, truncation=True, max_length=self.max_len,
            padding="max_length", return_tensors="pt"
        )
        input_ids = enc["input_ids"][0]
        attention_mask = enc["attention_mask"][0]
        labels = input_ids.clone()
        labels[attention_mask == 0] = -100
        return {"input_ids": input_ids,
                "attention_mask": attention_mask,
                "labels": labels}

In [ ]:
from torch.utils.data import DataLoader
from torch.optim import AdamW

def train_toy_lora(task_name, pairs, epochs=15, lr=1e-3):
    model = get_peft_model(copy.deepcopy(base_model), lora_config)
    model.to(device)
    model.train()

    ds = ToyDataset(pairs, tokenizer)
    dl = DataLoader(ds, batch_size=2, shuffle=True)
    optim = AdamW(model.parameters(), lr=lr)

    for epoch in range(epochs):
        for batch in dl:
            batch = {k: v.to(device) for k, v in batch.items()}
            out = model(**batch)
            loss = out.loss
            loss.backward()
            optim.step()
            optim.zero_grad()
    print(f"[{task_name}] final loss: {loss.item():.4f}")

    lora_state = {k: v.detach().clone().cpu()
                  for k, v in model.state_dict().items() if "lora_" in k}
    return lora_state

In [ ]:
import torch.nn.functional as F
import gc

def cleanup():
    gc.collect()
    torch.cuda.empty_cache()

def flatten_lora_deltas(lora_state):
    modules = {}
    for k, v in lora_state.items():
        prefix = k.rsplit(".lora_", 1)[0]
        kind = "A" if "lora_A" in k else "B"
        modules.setdefault(prefix, {})[kind] = v
    flat_vectors = []
    for prefix, d in sorted(modules.items()):
        if "A" in d and "B" in d:
            delta = d["B"].float() @ d["A"].float()
            flat_vectors.append(delta.flatten())
    return torch.cat(flat_vectors)

def lora_similarity(lora_state_1, lora_state_2):
    v1 = flatten_lora_deltas(lora_state_1)
    v2 = flatten_lora_deltas(lora_state_2)
    return F.cosine_similarity(v1.unsqueeze(0), v2.unsqueeze(0)).item()

def merge_loras(stored_lora, incoming_lora, n_merged_so_far):
    merged = {}
    for k in stored_lora.keys():
        merged[k] = (incoming_lora[k] + n_merged_so_far * stored_lora[k]) / (n_merged_so_far + 1)
    return merged

print("K-Merge core functions ready.")

In [ ]:
def load_lora_state_into_model(lora_state):
    model = get_peft_model(copy.deepcopy(base_model), lora_config)
    model.to(device)
    sd = model.state_dict()
    for k, v in lora_state.items():
        if k in sd:
            sd[k] = v.to(device).to(sd[k].dtype)
    model.load_state_dict(sd)
    model.eval()
    return model

def generate(model, prompt, max_new_tokens=30):
    inputs = tokenizer(prompt + "\nAnswer:", return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("Inference functions ready.")

In [ ]:
import difflib
from collections import Counter

def clean(s):
    return "".join(c.lower() for c in s if c.isalnum() or c.isspace()).split()

def score(pred, target):
    p = clean(pred)
    t = clean(target)
    if not p or not t:
        return 0
    common = set(p) & set(t)
    if not common:
        return 0
    overlap = sum(min(p.count(w), t.count(w)) for w in common)
    prec = overlap / len(p)
    rec = overlap / len(t)
    return 2 * prec * rec / (prec + rec) if (prec + rec) else 0

def get_ngrams(tokens, n):
    return Counter([tuple(tokens[i:i+n])
                    for i in range(len(tokens)-n+1)])

def gleu_score(pred, target):
    pred_tokens = clean(pred)
    target_tokens = clean(target)
    if not pred_tokens or not target_tokens:
        return 0.0
    scores = []
    for n in [1, 2]:
        pred_ng = get_ngrams(pred_tokens, n)
        target_ng = get_ngrams(target_tokens, n)
        if not pred_ng or not target_ng:
            scores.append(0.0)
            continue
        overlap = sum((pred_ng & target_ng).values())
        precision = overlap / max(sum(pred_ng.values()), 1)
        recall = overlap / max(sum(target_ng.values()), 1)
        if precision + recall == 0:
            scores.append(0.0)
        else:
            scores.append(2 * precision * recall / (precision + recall))
    return round(sum(scores) / len(scores), 3)

def run_eval_detailed(model, eval_set, tag=""):
    rows = []
    for prompt, target in eval_set:
        out = generate(model, prompt)
        rows.append({
            "task": tag,
            "input": prompt,
            "expected": target,
            "prediction": out,
            "exact_match": 1.0 if clean(out) == clean(target) else 0.0,
            "edit_acc": round(difflib.SequenceMatcher(
                None, out.strip().lower(),
                target.strip().lower()).ratio(), 3),
            "f1": round(score(out, target), 3),
            "gleu": gleu_score(out, target),
        })
    n = len(rows)
    summary = {
        "avg_exact_match": round(sum(r["exact_match"] for r in rows)/n, 3),
        "avg_edit_acc":    round(sum(r["edit_acc"]    for r in rows)/n, 3),
        "avg_f1":          round(sum(r["f1"]          for r in rows)/n, 3),
        "avg_gleu":        round(sum(r["gleu"]        for r in rows)/n, 3),
    }
    return rows, summary

print("All 4 metrics ready: exact_match, edit_acc, f1, gleu")

In [ ]:
import random as rnd

en_subjects = ["He", "She", "It", "The dog", "The cat", "My sister",
               "Her brother", "The teacher", "The doctor", "The boy"]
en_verb_pairs = [("like","likes"),("want","wants"),("love","loves"),
                  ("know","knows"),("go","goes"),("do","does"),
                  ("play","plays"),("eat","eats")]
en_objects = ["the book","the game","the food","the answer","the song"]

def build_en_examples():
    rows = []
    for subj in en_subjects:
        for base, s_form in en_verb_pairs:
            for obj in en_objects:
                rows.append((f"Fix the grammar: {subj} {base} {obj}.",
                              f"{subj} {s_form} {obj}."))
    return rows

de_subjects = ["Er","Sie","Es","Mein Bruder","Meine Schwester","Der Lehrer",
               "Die Ärztin","Der Hund","Die Katze","Mein Freund","Der Fahrer",
               "Die Köchin","Der Student","Das Kind","Der Nachbar"]
de_objects = ["ein Auto","einen Hund","eine Katze","ein Buch","ein Fahrrad",
              "einen Bruder","eine Schwester","ein Haus","einen Ball","eine Tasche"]

def build_de_examples():
    rows = []
    for subj in de_subjects:
        for obj in de_objects:
            rows.append((f"Fix the grammar: {subj} haben {obj}.",
                          f"{subj} hat {obj}."))
    return rows

fr_subjects = ["Il","Elle","Mon frère","Ma sœur","Le professeur",
               "Le médecin","Le chien","Le chat","Mon ami","La fille"]
fr_verb_pairs = [("manger","mange"),("aimer","aime"),("parler","parle"),
                  ("chanter","chante"),("jouer","joue"),("regarder","regarde"),
                  ("écouter","écoute"),("chercher","cherche")]
fr_objects = ["le repas","la musique","le film","le jeu","le livre"]

def build_fr_examples():
    rows = []
    for subj in fr_subjects:
        for base, s_form in fr_verb_pairs:
            for obj in fr_objects:
                rows.append((f"Fix the grammar: {subj} {base} {obj}.",
                              f"{subj} {s_form} {obj}."))
    return rows

rnd.seed(42)
en_pool = list(set(build_en_examples()))
de_pool = list(set(build_de_examples()))
fr_pool = list(set(build_fr_examples()))
rnd.shuffle(en_pool)
rnd.shuffle(de_pool)
rnd.shuffle(fr_pool)

print("EN pool:", len(en_pool))
print("DE pool:", len(de_pool))
print("FR pool:", len(fr_pool))

In [ ]:
import pandas as pd

EVAL_SIZE_100 = 100
en_eval_100 = en_pool[:EVAL_SIZE_100]
de_eval_100 = de_pool[:EVAL_SIZE_100]
fr_eval_100 = fr_pool[:EVAL_SIZE_100]

eval_map_100 = {
    "grammar_en": en_eval_100,
    "grammar_de": de_eval_100,
    "grammar_fr": fr_eval_100,
}

print("eval_map_100 ready:", {k: len(v) for k, v in eval_map_100.items()})

for lang, pool in [("en", en_eval_100),
                   ("de", de_eval_100),
                   ("fr", fr_eval_100)]:
    train_prompts = set(p for p, _ in toy_tasks[f"grammar_{lang}"])
    overlap = train_prompts & set(p for p, _ in pool)
    print(f"Overlap {lang}: {len(overlap)}")

In [ ]:
import os

def train_and_save_adapters(seed, tasks, save_dir="saved_loras"):
    os.makedirs(save_dir, exist_ok=True)
    random.seed(seed)
    torch.manual_seed(seed)
    loras = {}
    for name, pairs in tasks.items():
        path = f"{save_dir}/seed{seed}_{name}.pt"
        if os.path.exists(path):
            loras[name] = torch.load(path)
            print(f"Loaded saved: {path}")
        else:
            loras[name] = train_toy_lora(name, pairs)
            torch.save(loras[name], path)
            print(f"Trained and saved: {path}")
    return loras

all_saved_loras = {}
for seed in [0, 1, 2]:
    all_saved_loras[seed] = train_and_save_adapters(
        seed,
        {k: toy_tasks[k] for k in
         ["grammar_en", "grammar_de", "grammar_fr"]}
    )

print("All adapters trained and saved.")
print("Seeds available:", list(all_saved_loras.keys()))

In [ ]:
def run_arrival_sequence_full(seed, order_names, eval_map, K=2):
    loras = {name: {k: v.clone()
                    for k, v in all_saved_loras[seed][name].items()}
             for name in order_names}

    lang_map = {"grammar_en": "en",
                "grammar_de": "de",
                "grammar_fr": "fr"}
    collection = {}
    history = {}
    step_log = []
    all_pred_rows = []

    for t, name in enumerate(order_names):
        incoming = loras[name]
        merge_event = None

        if len(collection) < K:
            slot = name
            collection[slot] = {k: v.clone() for k, v in incoming.items()}
            history[slot] = {name}
        else:
            sims = {s: lora_similarity(collection[s], incoming)
                    for s in collection}
            best_slot = max(sims, key=sims.get)
            merge_event = {
                "merged_into": best_slot,
                "incoming": name,
                "all_similarities": {k: round(v, 4)
                                     for k, v in sims.items()},
                "selected_similarity": round(sims[best_slot], 4),
            }
            n_prev = len(history[best_slot])
            collection[best_slot] = merge_loras(
                collection[best_slot], incoming, n_prev)
            history[best_slot].add(name)

        task_to_slot_now = {}
        for slot, tasks in history.items():
            for task in tasks:
                task_to_slot_now[task] = slot

        seen = order_names[:t+1]
        per_lang_scores = {}
        for task in seen:
            slot = task_to_slot_now[task]
            model = load_lora_state_into_model(collection[slot])
            rows, summary = run_eval_detailed(
                model, eval_map[task],
                tag=f"seed{seed}_step{t}_{lang_map[task]}"
            )
            for r in rows:
                r.update({"seed": seed,
                           "order": "->".join(order_names),
                           "step": t,
                           "task": task})
            all_pred_rows.extend(rows)
            per_lang_scores[lang_map[task]] = summary
            del model
            cleanup()

        step_log.append({
            "seed": seed,
            "order": "->".join(order_names),
            "step": t,
            "arrived": name,
            "merge_event": merge_event,
            "per_lang_f1": {k: v["avg_f1"]
                            for k, v in per_lang_scores.items()},
            "per_lang_exact": {k: v["avg_exact_match"]
                               for k, v in per_lang_scores.items()},
            "per_lang_edit": {k: v["avg_edit_acc"]
                              for k, v in per_lang_scores.items()},
            "per_lang_gleu": {k: v["avg_gleu"]
                              for k, v in per_lang_scores.items()},
            "avg_f1": round(
                sum(v["avg_f1"]
                    for v in per_lang_scores.values()) /
                len(per_lang_scores), 3),
            "worst_f1": round(
                min(v["avg_f1"]
                    for v in per_lang_scores.values()), 3),
        })

    return step_log, all_pred_rows

print("Corrected run function ready.")

In [ ]:
def run_pairwise_merge(seed, task_a, task_b, eval_map):
    lora_a = all_saved_loras[seed][task_a]
    lora_b = all_saved_loras[seed][task_b]
    lang_a = task_a.split("_")[1]
    lang_b = task_b.split("_")[1]

    sim = lora_similarity(lora_a, lora_b)

    model_a = load_lora_state_into_model(lora_a)
    _, sum_a_before = run_eval_detailed(model_a, eval_map[task_a])
    del model_a; cleanup()

    model_b = load_lora_state_into_model(lora_b)
    _, sum_b_before = run_eval_detailed(model_b, eval_map[task_b])
    del model_b; cleanup()

    merged_ab = merge_loras(lora_b, lora_a, 1)
    model_ab = load_lora_state_into_model(merged_ab)
    _, sum_a_after_ab = run_eval_detailed(model_ab, eval_map[task_a])
    _, sum_b_after_ab = run_eval_detailed(model_ab, eval_map[task_b])
    del model_ab; cleanup()

    merged_ba = merge_loras(lora_a, lora_b, 1)
    model_ba = load_lora_state_into_model(merged_ba)
    _, sum_a_after_ba = run_eval_detailed(model_ba, eval_map[task_a])
    _, sum_b_after_ba = run_eval_detailed(model_ba, eval_map[task_b])
    del model_ba; cleanup()

    tensor_diff = max(
        abs(float(v1) - float(v2))
        for k in merged_ab
        for v1, v2 in zip(
            merged_ab[k].flatten().tolist(),
            merged_ba[k].flatten().tolist()
        )
    )

    return {
        "seed": seed,
        "pair": f"{lang_a}-{lang_b}",
        "similarity": round(sim, 4),
        f"{lang_a}_before": sum_a_before["avg_f1"],
        f"{lang_b}_before": sum_b_before["avg_f1"],
        f"{lang_a}_after_AB": sum_a_after_ab["avg_f1"],
        f"{lang_b}_after_AB": sum_b_after_ab["avg_f1"],
        f"{lang_a}_after_BA": sum_a_after_ba["avg_f1"],
        f"{lang_b}_after_BA": sum_b_after_ba["avg_f1"],
        f"change_{lang_a}_AB": round(
            sum_a_after_ab["avg_f1"] - sum_a_before["avg_f1"], 3),
        f"change_{lang_b}_AB": round(
            sum_b_after_ab["avg_f1"] - sum_b_before["avg_f1"], 3),
        "max_tensor_diff": round(tensor_diff, 6),
        "order_invariant": tensor_diff < 1e-4,
    }

print("Pairwise merge function ready.")

In [ ]:
pairs = [
    ("grammar_en", "grammar_de"),
    ("grammar_en", "grammar_fr"),
    ("grammar_de", "grammar_fr"),
]

pairwise_results = []
for seed in [0, 1, 2]:
    for task_a, task_b in pairs:
        result = run_pairwise_merge(seed, task_a, task_b, eval_map_100)
        pairwise_results.append(result)
        print(result)

pd.DataFrame(pairwise_results).to_csv(
    "pairwise_independent_merges.csv", index=False)
print("saved pairwise_independent_merges.csv")

In [ ]:
orders_to_test = [
    ["grammar_en", "grammar_de", "grammar_fr"],
    ["grammar_fr", "grammar_de", "grammar_en"],
    ["grammar_de", "grammar_fr", "grammar_en"],
]

corrected_logs = []
corrected_preds = []

for seed in [0, 1, 2]:
    for order in orders_to_test:
        log, preds = run_arrival_sequence_full(
            seed, order, eval_map_100, K=2)
        corrected_logs.extend(log)
        corrected_preds.extend(preds)
        print(f"\nseed={seed}, order={'->'.join(order)}")
        for row in log:
            print(f"  step {row['step']}: arrived={row['arrived']}")
            if row['merge_event']:
                print(f"    merge: {row['merge_event']}")
            print(f"    per_lang_f1: {row['per_lang_f1']}")
            print(f"    avg={row['avg_f1']}, worst={row['worst_f1']}")

pd.DataFrame(corrected_preds).to_csv(
    "corrected_sweep_predictions.csv", index=False)
pd.DataFrame(corrected_logs).to_csv(
    "corrected_sweep_logs.csv", index=False)
print("\nsaved all CSVs")

In [ ]:
df = pd.DataFrame(pairwise_results)

print("=== PAIRWISE SIMILARITY TABLE ===")
print(df[["seed","pair","similarity"]].to_string(index=False))

print("\n=== AVERAGE SIMILARITY BY PAIR ===")
print(df.groupby("pair")["similarity"].mean().round(4))

print("\n=== ORDER INVARIANCE CHECK ===")
print(df[["seed","pair","max_tensor_diff",
          "order_invariant"]].to_string(index=False))

In [ ]:
print("=== PAIRWISE MERGE RESULTS ===")
show_cols = ["seed","pair","similarity"]
for lang in ["en","de","fr"]:
    for col in [f"{lang}_before",
                f"{lang}_after_AB",
                f"change_{lang}_AB"]:
        if col in df.columns:
            show_cols.append(col)

print(df[[c for c in show_cols
          if c in df.columns]].to_string(index=False))

print("\n=== AVERAGE CHANGE BY PAIR ===")
change_cols = [c for c in df.columns if c.startswith("change_")]
print(df.groupby("pair")[change_cols].mean().round(3))

print("\n=== CORRECTED SWEEP SUMMARY ===")
if corrected_logs:
    final_steps = [r for r in corrected_logs if r["step"] == 2]
    for r in final_steps:
        print(f"seed={r['seed']} order={r['order']} "
              f"avg={r['avg_f1']} worst={r['worst_f1']} "
              f"per_lang={r['per_lang_f1']}")

In [ ]:
# Verify current state
print("Base model:", MODEL_NAME)
print("Seeds in all_saved_loras:", list(all_saved_loras.keys()))

# Check if saved files exist
import os
save_dir = "saved_loras"
for seed in [0,1,2]:
    for lang in ["en","de","fr"]:
        path = f"{save_dir}/seed{seed}_grammar_{lang}.pt"
        exists = os.path.exists(path)
        print(f"{path}: {'EXISTS' if exists else 'MISSING'}")

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token

from huggingface_hub import login
login(token=hf_token)
print("HuggingFace login successful")

In [ ]:
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

gc.collect()
torch.cuda.empty_cache()

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Loading:", MODEL_NAME)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)

print("Loaded:", MODEL_NAME)
print("Params:", round(sum(p.numel() for p in base_model.parameters())/1e6, 1), "M")

In [ ]:
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

gc.collect()
torch.cuda.empty_cache()

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Loading:", MODEL_NAME)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)

print("Loaded:", MODEL_NAME)
print("Params:", round(
    sum(p.numel() for p in base_model.parameters())/1e6, 1), "M")

In [ ]:
toy_tasks_large = {
    "grammar_en": [
        ("Fix the grammar: She go to school every day.", "She goes to school every day."),
        ("Fix the grammar: He don't like apples.", "He doesn't like apples."),
        ("Fix the grammar: They was late.", "They were late."),
        ("Fix the grammar: I has a car.", "I have a car."),
        ("Fix the grammar: We was happy yesterday.", "We were happy yesterday."),
        ("Fix the grammar: He like pizza very much.", "He likes pizza very much."),
        ("Fix the grammar: She don't know the answer.", "She doesn't know the answer."),
        ("Fix the grammar: The dog run fast.", "The dog runs fast."),
        ("Fix the grammar: My friend want to come.", "My friend wants to come."),
        ("Fix the grammar: The teacher go home early.", "The teacher goes home early."),
    ],
    "grammar_de": [
        ("Fix the grammar: Er gehen zur Schule.", "Er geht zur Schule."),
        ("Fix the grammar: Wir hat keine Zeit.", "Wir haben keine Zeit."),
        ("Fix the grammar: Sie ist mein Schwester.", "Sie ist meine Schwester."),
        ("Fix the grammar: Er haben ein Auto.", "Er hat ein Auto."),
        ("Fix the grammar: Du bist mein beste Freund.", "Du bist mein bester Freund."),
        ("Fix the grammar: Wir gehst ins Kino.", "Wir gehen ins Kino."),
        ("Fix the grammar: Das ist meine Bruder.", "Das ist mein Bruder."),
        ("Fix the grammar: Ich mag nicht Kaffee.", "Ich mag keinen Kaffee."),
        ("Fix the grammar: Sie kommt aus Deutschland nicht.", "Sie kommt nicht aus Deutschland."),
        ("Fix the grammar: Er haben kein Geld.", "Er hat kein Geld."),
    ],
    "grammar_fr": [
        ("Fix the grammar: Il ne sais pas.", "Il ne sait pas."),
        ("Fix the grammar: Nous avons manger.", "Nous avons mangé."),
        ("Fix the grammar: Elle est aller au marché.", "Elle est allée au marché."),
        ("Fix the grammar: Ils a fini le travail.", "Ils ont fini le travail."),
        ("Fix the grammar: Elle est parti hier.", "Elle est partie hier."),
        ("Fix the grammar: Tu as bien dormis?", "Tu as bien dormi?"),
        ("Fix the grammar: Nous avons vu le chien noire.", "Nous avons vu le chien noir."),
        ("Fix the grammar: Il fait ses devoir.", "Il fait ses devoirs."),
        ("Fix the grammar: Elle chante très bein.", "Elle chante très bien."),
        ("Fix the grammar: Je suis content de te voire.", "Je suis content de te voir."),
    ],
}

print("Larger toy tasks defined:")
for k, v in toy_tasks_large.items():
    print(f"  {k}: {len(v)} examples")

In [ ]:
all_saved_loras_backup = all_saved_loras
all_saved_loras = all_saved_loras_qwen15

pairs = [
    ("grammar_en", "grammar_de"),
    ("grammar_en", "grammar_fr"),
    ("grammar_de", "grammar_fr"),
]

qwen15_pairwise = []
for seed in [0, 1, 2]:
    for task_a, task_b in pairs:
        result = run_pairwise_merge(seed, task_a, task_b, eval_map_100)
        result["model"] = "Qwen2.5-1.5B"
        qwen15_pairwise.append(result)
        print(result)

pd.DataFrame(qwen15_pairwise).to_csv(
    "qwen15_pairwise_merges.csv", index=False)
print("saved qwen15_pairwise_merges.csv")

all_saved_loras = all_saved_loras_backup
print("Restored.")

In [ ]:
# Check what's available
checks = {
    "base_model": "base_model" in globals(),
    "all_saved_loras": "all_saved_loras" in globals(),
    "pairwise_results": "pairwise_results" in globals(),
    "eval_map_100": "eval_map_100" in globals(),
    "run_pairwise_merge": "run_pairwise_merge" in globals(),
    "corrected_logs": "corrected_logs" in globals(),
}
for k, v in checks.items():
    print(f"{k}: {'✅ available' if v else '❌ missing'}")

In [ ]:
print("toy_tasks_large" in globals())
print("toy_tasks_large defined:", "toy_tasks_large" in globals())
print("all_saved_loras defined:", "all_saved_loras" in globals())
print("train_and_save_adapters defined:", "train_and_save_adapters" in globals())

In [ ]:
QWEN15_DIR = "saved_loras_qwen15"

toy_tasks_backup = toy_tasks
toy_tasks = toy_tasks_large

all_saved_loras_qwen15 = {}
for seed in [0, 1, 2]:
    all_saved_loras_qwen15[seed] = train_and_save_adapters(
        seed,
        {k: toy_tasks[k] for k in
         ["grammar_en", "grammar_de", "grammar_fr"]},
        save_dir=QWEN15_DIR
    )

toy_tasks = toy_tasks_backup
print("Qwen2.5-1.5B adapters ready:", list(all_saved_loras_qwen15.keys()))

In [ ]:
all_saved_loras_backup = all_saved_loras
all_saved_loras = all_saved_loras_qwen15

pairs = [
    ("grammar_en", "grammar_de"),
    ("grammar_en", "grammar_fr"),
    ("grammar_de", "grammar_fr"),
]

qwen15_pairwise = []
for seed in [0, 1, 2]:
    for task_a, task_b in pairs:
        result = run_pairwise_merge(seed, task_a, task_b, eval_map_100)
        result["model"] = "Qwen2.5-1.5B"
        qwen15_pairwise.append(result)
        print(result)

pd.DataFrame(qwen15_pairwise).to_csv(
    "qwen15_pairwise_merges.csv", index=False)
print("saved qwen15_pairwise_merges.csv")

all_saved_loras = all_saved_loras_backup
print("Restored.")

In [ ]:
all_saved_loras = all_saved_loras_qwen15

orders_to_test = [
    ["grammar_en", "grammar_de", "grammar_fr"],
    ["grammar_fr", "grammar_de", "grammar_en"],
    ["grammar_de", "grammar_fr", "grammar_en"],
]

qwen15_logs = []
qwen15_preds = []

for seed in [0, 1, 2]:
    for order in orders_to_test:
        log, preds = run_arrival_sequence_full(
            seed, order, eval_map_100, K=2)
        for row in log:
            row["model"] = "Qwen2.5-1.5B"
        qwen15_logs.extend(log)
        qwen15_preds.extend(preds)
        print(f"\nseed={seed}, order={'->'.join(order)}")
        for row in log:
            print(f"  step {row['step']}: {row['arrived']}")
            if row['merge_event']:
                print(f"    merge: {row['merge_event']}")
            print(f"    per_lang_f1: {row['per_lang_f1']}")
            print(f"    avg={row['avg_f1']}, worst={row['worst_f1']}")

pd.DataFrame(qwen15_preds).to_csv(
    "qwen15_sweep_predictions.csv", index=False)
pd.DataFrame(qwen15_logs).to_csv(
    "qwen15_sweep_logs.csv", index=False)

all_saved_loras = all_saved_loras_backup
print("\nsaved Qwen1.5B sweep CSVs.")